In [ ]:
import pandas as pd

# load corpus

df = pd.read_csv(
    "telex_article_dataset_full.csv",
    low_memory=False
)

print("=" * 80)

print("rows")

print(len(df))

print("\ncolumns")

for c in df.columns:
    print(c)

print("\n" + "=" * 80)

# missing values

summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:

        empty = (

            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()

        )

    summary.append({

        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)

    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))

print("\n" + "=" * 80)

# important fields

important = [

    "article_url",
    "publication_date",
    "title",
    "article_text",
    "category"

]

print("\nimportant fields\n")

for field in important:

    if field in df.columns:

        print(f"{field:20} OK")

    else:

        print(f"{field:20} MISSING")

In [ ]:
russia_keywords = [

    r"\borosz\w*",
    r"\bkreml\w*",
    r"\bmoszkv\w*",
    r"\bputyin\w*",
    r"\blavrov\w*",
    r"\bpeszkov\w*",
    r"\bsojgu\w*",
    r"\bgeraszimov\w*",
    r"\bmedvegyev\w*",
    r"\bzaharova\w*",
    r"\bmisusztyin\w*",
    r"\bgazprom\w*",
    r"\brosznyefty\w*",
    r"\broszatom\w*",
    r"\btransznyefty\w*",
    r"\blukoil\w*",
    r"\bduma\w*",
    r"\bkrím\w*",
    r"\bukrajnai háború\b",
    r"\bagressziós háború\b"

]

In [ ]:
pattern = "|".join(russia_keywords)

russia_df = df[

    (

        df["title"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False
        )

    )

    |

    (

        df["article_text"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False
        )

    )

].copy()

print()

print("articles after filtering")

print(len(russia_df))

In [ ]:
keyword_counts = {}

text = (

    df["title"]
    .fillna("")
    + " "
    + df["article_text"]
    .fillna("")

)

for keyword in russia_keywords:

    keyword_counts[keyword] = (

        text
        .str.contains(
            keyword,
            case=False,
            na=False
        )
        .sum()

    )

keyword_counts = (

    pd.Series(keyword_counts)
    .sort_values(ascending=False)
    .reset_index()

)

keyword_counts.columns = [

    "keyword",
    "articles"

]

display(keyword_counts)

In [ ]:
old_df = pd.read_csv(
    "telex_ukraine_russia_high_recall.csv"
)

old_urls = set(old_df["article_url"])

new_urls = set(russia_df["article_url"])

print(f"old corpus: {len(old_urls):,}")

print(f"new corpus: {len(new_urls):,}")

In [ ]:
print(f"overlap: {len(old_urls & new_urls):,}")

print(f"only old: {len(old_urls - new_urls):,}")

print(f"only new: {len(new_urls - old_urls):,}")

In [ ]:
russia_df.to_csv(

    "telex_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("final corpus saved")

print(len(russia_df))

In [ ]:
import os
import hashlib
import pandas as pd

# paths

new_corpus = (
    "telex_SECONDROUND_final_articles.csv"
)

old_corpus = (
    "telex_ukraine_russia_with_images_cleaned.csv"
)

image_folder = (
    "."
)

missing_file = (
    "telex_overlap_missing_images.csv"
)

# load datasets

new_df = pd.read_csv(
    new_corpus,
    low_memory=False
)

old_df = pd.read_csv(
    old_corpus,
    low_memory=False
)

# keep only articles already present in the old corpus

new_urls = set(
    new_df["article_url"]
)

overlap_df = old_df[
    old_df["article_url"].isin(new_urls)
].copy()

print("articles in overlap")

print(len(overlap_df))

# collect image urls

image_urls = set()

for _, row in overlap_df.iterrows():

    og_image = str(
        row.get("og_image", "")
    ).strip()

    if og_image.startswith("http"):

        image_urls.add(
            og_image
        )

    body_images = str(
        row.get("body_images", "")
    ).strip()

    if body_images:

        for url in body_images.split("|"):

            url = url.strip()

            if url.startswith("http"):

                image_urls.add(
                    url
                )

print()

print("unique image urls")

print(len(image_urls))

# check which images already exist

existing = 0

missing = []

for url in image_urls:

    filename = (

        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()

        + ".jpg"

    )

    filepath = os.path.join(
        image_folder,
        filename
    )

    if os.path.exists(filepath):

        existing += 1

    else:

        missing.append({

            "image_url": url,
            "filename": filename

        })

print()

print("images already downloaded")

print(existing)

print()

print("images missing")

print(len(missing))

# save missing images

missing_df = pd.DataFrame(
    missing
)

missing_df.to_csv(

    missing_file,

    index=False,

    encoding="utf-8-sig"

)

print()

print("saved")

print(missing_file)

In [ ]:
overlap_df = old_df[
    old_df["article_url"].isin(new_urls)
].copy()

print("articles in overlap")
print(len(overlap_df))

# collect image urls

articles_with_og = 0

articles_with_body = 0

total_image_urls = 0

unique_image_urls = set()

for _, row in overlap_df.iterrows():

    og_image = str(
        row.get("og_image", "")
    ).strip()

    if og_image.startswith("http"):

        articles_with_og += 1

        total_image_urls += 1

        unique_image_urls.add(
            og_image
        )

    body_images = row.get(
        "body_images",
        "[]"
    )

    try:

        body_images = ast.literal_eval(
            str(body_images)
        )

    except:

        body_images = []

    body_urls = [

        url

        for url in body_images

        if isinstance(url, str)
        and url.startswith("http")

    ]

    if len(body_urls) > 0:

        articles_with_body += 1

    total_image_urls += len(
        body_urls
    )

    unique_image_urls.update(
        body_urls
    )

print()

print("articles with og image")
print(articles_with_og)

print()

print("articles with body images")
print(articles_with_body)

print()

print("total image urls")
print(total_image_urls)

print()

print("unique image urls")
print(len(unique_image_urls))

# check downloaded images

downloaded = 0

missing = []

for url in unique_image_urls:

    filename = (

        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()

        + ".jpg"

    )

    filepath = os.path.join(
        image_folder,
        filename
    )

    if os.path.exists(filepath):

        downloaded += 1

    else:

        missing.append({

            "image_url": url,
            "filename": filename

        })

print()

print("images already downloaded")
print(downloaded)

print()

print("images missing")
print(len(missing))

# save missing list

pd.DataFrame(
    missing
).to_csv(

    missing_file,

    index=False,

    encoding="utf-8-sig"

)

print()

print("saved")
print(missing_file)

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# settings

input_file = (
    "telex_SECONDROUND_final_articles.csv"
)

old_file = (
    "telex_ukraine_russia_with_images_cleaned.csv"
)

output_file = (
    "telex_SECONDROUND_final_articles_with_images.csv"
)

max_workers = 8

# load datasets

new_df = pd.read_csv(
    input_file,
    low_memory=False
)

old_df = pd.read_csv(
    old_file,
    low_memory=False
)

new_df["article_url"] = (

    new_df["article_url"]
    .astype(str)
    .str.strip()

)

old_df["article_url"] = (

    old_df["article_url"]
    .astype(str)
    .str.strip()

)

print("\nrows\n")

print(len(new_df))

# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}

# session

session = requests.Session()

session.headers.update(headers)

# extractor

def extract_images(url):

    try:

        response = session.get(
            url,
            timeout=30
        )

        if response.status_code != 200:

            return None, [], 0, False

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        # og image

        og_image = None

        tag = soup.find(
            "meta",
            property="og:image"
        )

        if tag:

            og_image = tag.get(
                "content"
            )

        # body images

        body_images = []

        for img in soup.find_all("img"):

            src = img.get("src")

            if not src:
                continue

            src = str(src)

            if (
                "hero-full" in src
                or "cikktorzs" in src
            ):

                body_images.append(src)

        body_images = list(
            dict.fromkeys(body_images)
        )

        return (

            og_image,

            str(body_images),

            len(body_images),

            len(body_images) > 0

        )

    except:

        return None, "[]", 0, False

# merge old image information

columns = [

    "article_url",
    "og_image",
    "body_images",
    "body_image_count",
    "has_body_images"

]

old_images = old_df[
    columns
]


df = new_df.merge(

    old_images,

    on="article_url",

    how="left"

)

df["body_images"] = df["body_images"].astype(object)
old_urls = set(
    old_df["article_url"]
)

mask = (

    ~df["article_url"].isin(
        old_urls
    )

)

print("\narticles already processed\n")

print((~mask).sum())

print("\narticles to process\n")

print(mask.sum())

# prepare urls

indexes = list(
    df[mask].index
)

urls = list(
    df.loc[
        mask,
        "article_url"
    ]
)

# process in parallel

results = list(

    tqdm(

        ThreadPoolExecutor(
            max_workers=max_workers
        ).map(
            extract_images,
            urls
        ),

        total=len(urls)

    )

)

# save results

for idx, result in zip(indexes, results):

    og_image, body_images, count, has_images = result

    df.at[
        idx,
        "og_image"
    ] = og_image

    df.at[
        idx,
        "body_images"
    ] = str(body_images)

    df.at[
        idx,
        "body_image_count"
    ] = count

    df.at[
        idx,
        "has_body_images"
    ] = has_images

# diagnostics

print("\nog images found\n")

print(
    df["og_image"]
    .notna()
    .sum()
)

print("\nmissing og images\n")

print(
    df["og_image"]
    .isna()
    .sum()
)

print("\narticles with body images\n")

print(
    df["has_body_images"]
    .value_counts()
)

print("\nbody image count stats\n")

print(
    df["body_image_count"]
    .describe()
)

# save

df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)

print("\nsaved\n")

print(output_file)

print("\nfinished")

In [ ]:
df

In [ ]:
old_df = pd.read_csv(
    "telex_ukraine_russia_with_images_cleaned.csv"
)

print()

print("old")

print(
    old_df["og_image"]
    .notna()
    .sum()
)

print()

print("new")

print(
    df["og_image"]
    .notna()
    .sum()
)

In [ ]:
new_only = df[
    ~df["article_url"].isin(
        old_df["article_url"]
    )
]

print(len(new_only))

print()

print(
    new_only["og_image"]
    .notna()
    .sum()
)

In [ ]:
import ast

new_only = df[
    ~df["article_url"].isin(
        old_df["article_url"]
    )
].copy()

new_only["body_image_count"] = new_only["body_images"].apply(

    lambda x: len(ast.literal_eval(str(x)))
)

print()

print("new articles")

print(len(new_only))

print()

print("articles with body images")

print(
    (new_only["body_image_count"] > 0).sum()
)

print()

print("total body images")

print(
    new_only["body_image_count"].sum()
)

print()

print("average body images")

print(
    new_only["body_image_count"].mean()
)

In [ ]:
import os
import ast
import time
import hashlib
import requests
import pandas as pd

from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# paths

new_file = (
    "telex_SECONDROUND_final_articles_with_images.csv"
)

old_file = (
    "telex_ukraine_russia_with_images_cleaned.csv"
)

output_folder = (
    "telex_images_SECONDROUND"
)

failed_file = (
    "telex_SECONDROUND_failed_images.csv"
)

os.makedirs(
    output_folder,
    exist_ok=True
)

# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}

max_workers = 8

max_retries = 3

# load datasets

new_df = pd.read_csv(
    new_file,
    low_memory=False
)

old_df = pd.read_csv(
    old_file,
    low_memory=False
)

new_df["article_url"] = (

    new_df["article_url"]
    .astype(str)
    .str.strip()

)

old_df["article_url"] = (

    old_df["article_url"]
    .astype(str)
    .str.strip()

)

# keep only new articles

old_urls = set(
    old_df["article_url"]
)

new_only = new_df[
    ~new_df["article_url"].isin(
        old_urls
    )
].copy()

print()

print("new articles")

print(len(new_only))

# collect image urls

image_urls = set()

for _, row in new_only.iterrows():

    og = str(
        row.get(
            "og_image",
            ""
        )
    ).strip()

    if og.startswith("http"):

        image_urls.add(
            og
        )

    try:

        body = ast.literal_eval(

            str(
                row.get(
                    "body_images",
                    "[]"
                )
            )

        )

    except:

        body = []

    for url in body:

        if (
            isinstance(url, str)
            and
            url.startswith("http")
        ):

            image_urls.add(
                url
            )

image_urls = sorted(image_urls)

print()

print("unique image urls")

print(len(image_urls))

# download function

def download_image(url):

    session = requests.Session()

    session.headers.update(
        headers
    )

    filename = (

        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()

        + ".jpg"

    )

    filepath = os.path.join(

        output_folder,

        filename

    )

    if os.path.exists(
        filepath
    ):

        return "exists", None

    for _ in range(
        max_retries
    ):

        try:

            response = session.get(

                url,

                timeout=60

            )

            if response.status_code == 200:

                with open(

                    filepath,

                    "wb"

                ) as f:

                    f.write(
                        response.content
                    )

                return "downloaded", None

        except:

            pass

        time.sleep(2)

    return "failed", {

        "image_url": url,

        "filename": filename

    }

# download images

downloaded = 0

already_exists = 0

failed = []

with ThreadPoolExecutor(

    max_workers=max_workers

) as executor:

    results = list(

        tqdm(

            executor.map(

                download_image,

                image_urls

            ),

            total=len(
                image_urls
            )

        )

    )

for status, info in results:

    if status == "downloaded":

        downloaded += 1

    elif status == "exists":

        already_exists += 1

    else:

        failed.append(
            info
        )

# save failed downloads

pd.DataFrame(
    failed
).to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)

print()

print("downloaded")

print(downloaded)

print()

print("already existed")

print(already_exists)

print()

print("failed")

print(len(failed))

print()

print("saved folder")

print(output_folder)

print()

print("failed file")

print(failed_file)

In [ ]:
import os
import ast
import shutil
import hashlib
import pandas as pd

from tqdm import tqdm

# paths

new_file = (
    "telex_SECONDROUND_final_articles_with_images.csv"
)

old_file = (
    "telex_ukraine_russia_with_images_cleaned.csv"
)

source_folder = (
    "."
)

destination_folder = (
    "telex_images_SECONDROUND"
)

missing_file = (
    "telex_SECONDROUND_missing_overlap_images.csv"
)

# load datasets

new_df = pd.read_csv(
    new_file,
    low_memory=False
)

old_df = pd.read_csv(
    old_file,
    low_memory=False
)

new_df["article_url"] = (
    new_df["article_url"]
    .astype(str)
    .str.strip()
)

old_df["article_url"] = (
    old_df["article_url"]
    .astype(str)
    .str.strip()
)

# keep only overlap

new_urls = set(
    new_df["article_url"]
)

overlap_df = old_df[
    old_df["article_url"].isin(
        new_urls
    )
].copy()

print()

print("overlap articles")

print(len(overlap_df))

# collect filenames

filenames = set()

for _, row in overlap_df.iterrows():

    og = str(
        row.get(
            "og_image",
            ""
        )
    ).strip()

    if og.startswith("http"):

        filenames.add(

            hashlib.md5(
                og.encode("utf-8")
            ).hexdigest()

            + ".jpg"

        )

    try:

        body = ast.literal_eval(
            str(
                row.get(
                    "body_images",
                    "[]"
                )
            )
        )

    except:

        body = []

    for url in body:

        if (
            isinstance(url, str)
            and
            url.startswith("http")
        ):

            filenames.add(

                hashlib.md5(
                    url.encode("utf-8")
                ).hexdigest()

                + ".jpg"

            )

print()

print("unique images needed")

print(len(filenames))

# copy images

copied = 0

already_exists = 0

missing = []

for filename in tqdm(
    sorted(filenames)
):

    source = os.path.join(
        source_folder,
        filename
    )

    destination = os.path.join(
        destination_folder,
        filename
    )

    if os.path.exists(
        destination
    ):

        already_exists += 1

        continue

    if not os.path.exists(
        source
    ):

        missing.append(
            filename
        )

        continue

    shutil.copy2(
        source,
        destination
    )

    copied += 1

# save missing

pd.DataFrame({

    "filename": missing

}).to_csv(

    missing_file,

    index=False,

    encoding="utf-8-sig"

)

print()

print("copied")

print(copied)

print()

print("already existed")

print(already_exists)

print()

print("missing")

print(len(missing))

print()

print("saved folder")

print(destination_folder)

print()

print("missing file")

print(missing_file)

In [ ]:
import os
import ast
import hashlib
import pandas as pd

df = pd.read_csv(
    "telex_SECONDROUND_final_articles_with_images.csv",
    low_memory=False
)

IMAGE_FOLDER = "telex_images_SECONDROUND"

image_urls = []
image_filenames = []

missing_files = []

for _, row in df.iterrows():

    urls = []

    # ---------- og image ----------

    og = str(row.get("og_image", "")).strip()

    if og.startswith("http"):
        urls.append(og)

    # ---------- body images ----------

    try:
        body = ast.literal_eval(
            str(row.get("body_images", "[]"))
        )

        if isinstance(body, list):

            urls.extend(
                u.strip()
                for u in body
                if isinstance(u, str)
                and u.strip().startswith("http")
            )

    except:
        pass

    # remove duplicates

    urls = list(dict.fromkeys(urls))

    filenames = []

    for url in urls:

        filename = (
            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()
            + ".jpg"
        )

        filenames.append(filename)

        if not os.path.exists(
            os.path.join(
                IMAGE_FOLDER,
                filename
            )
        ):
            missing_files.append(
                {
                    "article_url": row["article_url"],
                    "image_url": url,
                    "filename": filename
                }
            )

    image_urls.append("|".join(urls))
    image_filenames.append("|".join(filenames))

df["image_url"] = image_urls
df["image_filename"] = image_filenames

print("Articles:", len(df))
print("Missing image files:", len(missing_files))

print()

print("Empty image_url:",
      (df["image_url"] == "").sum())

print("Empty image_filename:",
      (df["image_filename"] == "").sum())

missing_files = pd.DataFrame(missing_files)

display(missing_files.head())

In [ ]:
# total image urls
total_urls = (
    df["image_url"]
    .fillna("")
    .apply(lambda x: 0 if x.strip() == "" else len(x.split("|")))
    .sum()
)

# total image filenames
total_filenames = (
    df["image_filename"]
    .fillna("")
    .apply(lambda x: 0 if x.strip() == "" else len(x.split("|")))
    .sum()
)

print("Total image URLs:", total_urls)
print("Total image filenames:", total_filenames)

In [ ]:
# unique urls
unique_urls = set()

for x in df["image_url"].fillna(""):

    if x.strip():

        unique_urls.update(
            u.strip()
            for u in x.split("|")
            if u.strip()
        )

# unique filenames
unique_filenames = set()

for x in df["image_filename"].fillna(""):

    if x.strip():

        unique_filenames.update(
            f.strip()
            for f in x.split("|")
            if f.strip()
        )

print("Unique image URLs:", len(unique_urls))
print("Unique image filenames:", len(unique_filenames))

In [ ]:
import pandas as pd

df = pd.read_csv(
    "telex_SECONDROUND_final_articles_with_images.csv",
    low_memory=False
)

print(df.shape)
print(df.columns.tolist())

display(df.head())

In [ ]:
import os

# старый файл без изображений -> резервная копия
os.rename(
    "telex_SECONDROUND_final_articles.csv",
    "telex_SECONDROUND_final_articles_old_noimages.csv"
)

# файл с изображениями -> основной файл
os.rename(
    "telex_SECONDROUND_final_articles_with_images.csv",
    "telex_SECONDROUND_final_articles.csv"
)

print("Done.")